In [1]:
# libraries
import open3d as o3d
import trimesh
import numpy as np
from pathlib import Path
import os
import random
from preprocessing1 import *

%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# parameters
# sampling_type = 0
num_points = 2048
normalize = True
normalize_type = 0
# make_watertight = False
delete_interior_points = False
# keep_normals = False
# fix_normals = False
num_objects = 11
empty_the_output_folder = True

In [3]:
root = get_project_root()
print(f"Project Root identified as: {root}")

# delete the previous files in the output folder
folder_path = root / Path("data/preprocessed/preprocessing1")
# Iterate through all items in the folder
for file in folder_path.iterdir():
    if file.is_file():
        file.unlink()

mesh_path_list = get_random_shapenet_meshes(seed=42, num_objects=num_objects, root=root)
for path in mesh_path_list:
    print(path)
# path_str = "/home/nikola/Projects/tum-adlr-ss26-07/data/studentGrasping/student_grasps_v1/02876657/9fe7e6a7bf8ca964efad53eb3f0b36fa/3/mesh.obj"
# mesh_path_list = [Path(path_str)]


Project Root identified as: C:\Users\chris\Desktop\GraspDataset
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\03636649\77a7d38645738e2212c5719ce6179\0\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\4d4fc73864844dad1ceb7b8cc3792fd\8\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\02808440\bdacdeb89e174d743321831d2245cf06\0\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\03797390\67b9abb424cf22a22d7082a28b056a5\9\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\cc48fe97a95e8716ccaa5ad584801c3e\2\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\b2498accc1c3fe732db3066d0100ee4\7\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\a1275bd03ab15100f6dbe3dc17d6cdf7\2\mesh.obj
C:\Users\chris\Desktop\GraspDataset\data\stude

In [4]:
mesh_list = []
for path in mesh_path_list:
    mesh_list.append(trimesh.load(path, force='mesh', skip_materials=True))

In [5]:
from numpy import rint


for index, mesh in enumerate(mesh_list):
# Fix normals and winding order
    mesh.fix_normals()

    # Attempt to make it watertight (fills holes)
    mesh.fill_holes()

    # Remove zero-area triangles and duplicate vertices
    mesh.process(validate=True)

    # 1. Merge vertices that are at the exact same location
    mesh.merge_vertices()

    # 3. Remove "degenerate" faces (faces with zero area)
    mesh.remove_infinite_values()
    output_path = root / Path("data/preprocessed/preprocessing1") / f"mesh{index}.obj"
    mesh.export(output_path)  # Save the cleaned mesh for inspection

    print(f"Is watertight: {mesh.is_watertight}")

Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False
Is watertight: False


In [6]:
# Sample points from the surface
# returns a (point_num, 3) NumPy array
point_cloud_list = []
for mesh in mesh_list:
    points = mesh.sample(num_points)
    # Create a PointCloud object for Trimesh utilities
    point_cloud_list.append(trimesh.points.PointCloud(points))

In [7]:
if normalize:
    if normalize_type == 0:
        for point_cloud in point_cloud_list:
            point_cloud.vertices = normalize_points_max(point_cloud.vertices)
    elif normalize_type == 1:
        for point_cloud in point_cloud_list:
            point_cloud.vertices = normalize_statistically_points(point_cloud.vertices)

In [8]:
if delete_interior_points:
    for mesh in point_cloud_list:
        point_cloud.vertices = get_exterior_points(point_cloud.vertices)

In [9]:

for index, point_cloud in enumerate(point_cloud_list):
    # Ensure output directory exists and save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path = root / Path("data/preprocessed/preprocessing1") / f"pointcloud{index}.obj"
    point_cloud.export(output_path)
    print(f"Saved preprocessed point cloud to: {output_path}")

Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud0.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud1.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud2.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud3.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud4.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud5.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud6.obj
Saved preprocessed point cloud to: C:\Users\chris\Desktop\GraspDataset\data\preprocessed\preprocessing1\pointcloud7.obj
Saved preprocessed point cloud to: C:\Us

In [10]:
#Reconstruct from point cloud to mesh using ball pivoting

reconstructed_mesh_list = []
for index, point_cloud in enumerate(point_cloud_list):
    mesh = points_to_mesh_bpa(point_cloud.vertices, radius=None)
    reconstructed_mesh_list.append(mesh)
    save_path = root / Path("data/preprocessed/preprocessing1") / f"reconstructed_mesh{index}.obj"
    o3d.io.write_triangle_mesh(save_path, mesh)


[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
